# LeetCode 469 — Convex Polygon bằng Python

Link đề: https://leetcode.com/problems/convex-polygon/ · Độ khó: **Medium** · Chủ đề: Geometry, Math, Cross Product (tích có hướng), Polygon

Skeleton LeetCode:

```python
class Solution:
    def isConvex(self, points: List[List[int]]) -> bool:
        ...
```

Ghi chú: đây là bài **Premium** của LeetCode. Đề trong notebook được viết lại theo hiểu biết về bài, nên hãy đối chiếu lại đề gốc nếu bạn có tài khoản Premium. Mọi đáp án đều được kiểm chứng bằng code, kể cả bằng các cách định nghĩa "lồi" hoàn toàn độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho các điểm theo thứ tự nối thành một **đa giác đơn** (không tự cắt); hỏi đa giác đó có **lồi** (convex) hay không.
2. Nắm **tích có hướng 2 chiều** `a × b = ax*by - ay*bx`: dấu của nó cho biết **rẽ trái, rẽ phải hay thẳng hàng**; cùng toán tử `%` để đi vòng quanh danh sách (chỉ số quay vòng).
3. Phát hiện **bước đột phá**: đa giác đơn là lồi ⇔ **mọi chỗ rẽ cùng một hướng** (bỏ qua các điểm thẳng hàng), nên chỉ cần quét một vòng `O(n)`.
4. Cài đặt theo 4 hướng khác nhau: **mọi cạnh có các đỉnh còn lại cùng phía (`O(n²)`) → dấu tích có hướng của ba đỉnh liên tiếp (`O(n)`) → so diện tích đa giác với diện tích bao lồi → tổng góc ngoài tuyệt đối bằng 360°**.
5. Tránh các lỗi kinh điển: quên đỉnh vòng (chỉ số cuối), xử lý sai điểm thẳng hàng (tích bằng 0), đặt lại `pre` về 0, và hiểu vì sao điều kiện "đa giác đơn" là bắt buộc.

## 2. Đề bài & ví dụ

Cho mảng `points` gồm các điểm trên mặt phẳng `X-Y`, `points[i] = [xi, yi]`. Khi nối các điểm **tuần tự** (điểm cuối nối lại điểm đầu) ta được một đa giác.

Hãy trả về `true` nếu đa giác này **lồi**, ngược lại `false`.

Bạn được **đảm bảo** đa giác tạo bởi các điểm là một **đa giác đơn** (simple polygon): mỗi đỉnh có đúng hai cạnh gặp nhau, và các cạnh không cắt nhau ở chỗ nào khác.

**Ví dụ 1:** `points = [[0,0],[0,5],[5,5],[5,0]]` → `true` (hình vuông).

**Ví dụ 2:** `points = [[0,0],[0,10],[10,10],[10,0],[5,5]]` → `false` (đỉnh `[5,5]` lõm vào trong, hình có "vết lõm").

**Ràng buộc:** `3 <= points.length <= 10^4`; `points[i].length == 2`; `-10^4 <= xi, yi <= 10^4`; mọi điểm **khác nhau**.

(Một đa giác có các đỉnh thẳng hàng trên cùng cạnh vẫn có thể là lồi; điều đó sẽ quan trọng ở dưới.)

In [1]:
import math
import random
import time
from typing import List

examples = [
    ([[0, 0], [0, 5], [5, 5], [5, 0]], True),
    ([[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]], False),
]
for pts, expected in examples:
    print(pts, "->", expected)

[[0, 0], [0, 5], [5, 5], [5, 0]] -> True
[[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]] -> False


## 3. Kiến thức nền

### 3.1. Vector từ hai điểm

Từ điểm `A = (x1, y1)` đến `B = (x2, y2)` ta có **vector** `AB = (x2 - x1, y2 - y1)`: "đi sang phải bao nhiêu, đi lên bao nhiêu".

In [2]:
A, B = (1, 2), (4, 6)
AB = (B[0] - A[0], B[1] - A[1])
print("AB =", AB)          # (3, 4)

AB = (3, 4)


### 3.2. Tích có hướng 2 chiều (cross product) và dấu của nó

Với hai vector `u = (ux, uy)`, `v = (vx, vy)`: **`u × v = ux*vy - uy*vx`** (một số, không phải vector).

Ý nghĩa hình học: nó bằng diện tích **có dấu** (gấp đôi tam giác) của hình bình hành dựng trên `u` và `v`. Dấu cho biết hướng quay từ `u` sang `v`:

- `u × v > 0`: `v` nằm **bên trái** `u` (quay **ngược chiều kim đồng hồ**).
- `u × v < 0`: `v` nằm **bên phải** `u` (quay **cùng chiều kim đồng hồ**).
- `u × v = 0`: hai vector **song song / thẳng hàng**.

In [3]:
def cross_vec(u, v):
    return u[0] * v[1] - u[1] * v[0]

print(cross_vec((1, 0), (0, 1)))      #  1 > 0 : từ trục x quay lên trục y là ngược chiều kim đồng hồ (rẽ trái)
print(cross_vec((1, 0), (0, -1)))     # -1 < 0 : rẽ phải
print(cross_vec((2, 0), (5, 0)))      #  0 : cùng phương
print(cross_vec((3, 4), (6, 8)))      #  0 : song song

1
-1
0
0


### 3.3. Chiều rẽ của ba điểm `A → B → C`

Đi từ `A` đến `B` rồi tới `C`. Ta rẽ trái hay phải ở `B`? Lấy `u = B - A` và `v = C - A` (hay `C - B`, dấu giống nhau) rồi xét dấu `u × v`. Ta định nghĩa:

`cross(A, B, C) = (B - A) × (C - A)`.

Quy ước: `> 0` rẽ trái (counter-clockwise), `< 0` rẽ phải, `= 0` ba điểm thẳng hàng.

In [4]:
def cross(a, b, c):
    """(b - a) x (c - a): dương = rẽ trái tại b, âm = rẽ phải, 0 = thẳng hàng."""
    return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])

print(cross((0, 0), (4, 0), (4, 3)))      # 12 > 0  : đi sang phải rồi lên -> rẽ trái
print(cross((0, 0), (4, 0), (4, -3)))     # -12 < 0 : rẽ phải
print(cross((0, 0), (2, 0), (4, 0)))      # 0       : thẳng hàng

12
-12
0


### 3.4. Đa giác lồi là gì?

Đa giác **lồi** nếu nối hai điểm bất kỳ trong đa giác thì đoạn thẳng nằm trọn trong đa giác. Có nhiều cách nói tương đương (ta sẽ dùng nhiều cách để cài đặt):

1. Mọi góc trong ≤ 180°.
2. Đi vòng quanh biên, ta **luôn rẽ cùng một hướng** (luôn trái hoặc luôn phải), có thể đi thẳng.
3. Với mỗi cạnh, **mọi đỉnh còn lại nằm cùng một phía** của đường thẳng chứa cạnh đó.
4. Đa giác trùng với **bao lồi** (convex hull: đa giác lồi nhỏ nhất chứa mọi điểm) của chính các đỉnh của nó.

Lưu ý: đa giác lồi **có thể có đỉnh thẳng hàng** (ví dụ hình chữ nhật mà ta thêm một đỉnh ở giữa một cạnh): góc tại đỉnh đó đúng 180°, vẫn lồi.

### 3.5. Chỉ số quay vòng với `%`

Điểm sau điểm cuối là điểm đầu. Với `n` điểm, "điểm kế tiếp" của chỉ số `i` là `(i + 1) % n`: khi `i = n - 1` thì `(n) % n = 0`.

In [5]:
n = 5
for i in range(n):
    print(f"i={i}: kế tiếp={(i + 1) % n}, kế kế tiếp={(i + 2) % n}")

i=0: kế tiếp=1, kế kế tiếp=2
i=1: kế tiếp=2, kế kế tiếp=3
i=2: kế tiếp=3, kế kế tiếp=4
i=3: kế tiếp=4, kế kế tiếp=0
i=4: kế tiếp=0, kế kế tiếp=1


### 3.6. Công thức Shoelace (diện tích đa giác)

Với đa giác `P0, P1, ..., P(n-1)`, **gấp đôi diện tích** là `|Σ (xi*y(i+1) - x(i+1)*yi)|` (cộng theo vòng, chỉ số quay vòng). Làm việc với **gấp đôi diện tích** giúp dùng toàn số nguyên (không chia 2). Ta cần công thức này ở Cách 3.

In [6]:
def area2(poly):
    n = len(poly)
    return abs(sum(poly[i][0] * poly[(i + 1) % n][1] - poly[(i + 1) % n][0] * poly[i][1] for i in range(n)))

print(area2([[0, 0], [0, 5], [5, 5], [5, 0]]))      # 50 = 2 * 25
print(area2([[0, 0], [4, 0], [0, 3]]))              # 12 = 2 * 6

50
12


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng bạn **đi bộ dọc theo biên** của đa giác, mỗi đỉnh bạn phải xoay người. Trên hình vuông, ở **mọi** góc bạn đều xoay trái (nếu đi ngược chiều kim đồng hồ) hoặc đều xoay phải. Nhưng với hình có vết lõm, ở đỉnh lõm bạn phải xoay **ngược** hướng với các đỉnh khác.

**Tính tay Ví dụ 2:** `P0 = (0,0), P1 = (0,10), P2 = (10,10), P3 = (10,0), P4 = (5,5)`. Với mỗi đỉnh `i` ta tính `cross(Pi, P(i+1), P(i+2))` (chỗ rẽ tại `P(i+1)`):

| i | ba điểm | `u = P(i+1)-Pi` | `v = P(i+2)-Pi` | `u × v` | rẽ |
|---|---|---|---|---|---|
| 0 | (0,0),(0,10),(10,10) | (0,10) | (10,10) | `0*10 - 10*10 = -100` | phải |
| 1 | (0,10),(10,10),(10,0) | (10,0) | (10,-10) | `10*(-10) - 0*10 = -100` | phải |
| 2 | (10,10),(10,0),(5,5) | (0,-10) | (-5,-5) | `0*(-5) - (-5)(-10) = -50` | phải |
| 3 | (10,0),(5,5),(0,0) | (-5,5) | (-10,0) | `-5*0 - (-10)*5 = +50` | **trái** |
| 4 | (5,5),(0,0),(0,10) | (-5,-5) | (-5,5) | `-5*5 - (-5)(-5) = -50` | phải |

Bốn lần rẽ phải và **một lần rẽ trái** (tại `P4 = (5,5)`, đúng là đỉnh lõm) ⇒ không lồi ⇒ `false`.

**Với Ví dụ 1** (hình vuông) cả 4 lần đều cùng dấu ⇒ lồi.

In [7]:
def cac_tich(points):
    n = len(points)
    return [cross(points[i], points[(i + 1) % n], points[(i + 2) % n]) for i in range(n)]

print(cac_tich([[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]]))     # [-100, -100, -50, 50, -50]
print(cac_tich([[0, 0], [0, 5], [5, 5], [5, 0]]))                 # [-25, -25, -25, -25]

[-100, -100, -50, 50, -50]
[-25, -25, -25, -25]


## 5. Cách 1 — Brute Force / baseline: với mỗi cạnh, mọi đỉnh còn lại cùng một phía

**Ý tưởng:** dùng định nghĩa số 3 ở mục 3.4. Với mỗi cạnh `(Pi, P(i+1))`, ta xét mọi đỉnh `Pj`: tính `cross(Pi, P(i+1), Pj)`. Nếu có cả giá trị dương **và** âm thì hai đỉnh nằm hai phía của đường thẳng chứa cạnh ⇒ không lồi. Giá trị `0` (nằm trên đường thẳng) thì bỏ qua.

In [8]:
def brute_sides(points):
    n = len(points)
    for i in range(n):
        a, b = points[i], points[(i + 1) % n]
        co_duong = co_am = False
        for c in points:
            t = cross(a, b, c)
            if t > 0:
                co_duong = True
            elif t < 0:
                co_am = True
        if co_duong and co_am:          # có hai đỉnh ở hai phía của đường thẳng chứa cạnh (a, b)
            return False
    return True

for pts, expected in examples:
    print(pts, brute_sides(pts), expected)
    assert brute_sides(pts) == expected

[[0, 0], [0, 5], [5, 5], [5, 0]] True True
[[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]] False False


**Độ phức tạp:** hai vòng lặp lồng nhau → `O(n²)` thời gian, `O(1)` bộ nhớ. Với `n = 10^4` ta có `10^8` phép tính: quá chậm trong Python.

**Nút thắt:** ta kiểm tra **cả đa giác** cho mỗi cạnh trong khi dấu rẽ ở các đỉnh liền kề đã chứa đủ thông tin. Hãy đo.

In [9]:
def thin_rect(n, dent=False):
    """Hình chữ nhật mỏng có nhiều đỉnh thẳng hàng. dent=True: đẩy một đỉnh ở cạnh trên xuống thành vết lõm."""
    k = n // 2
    m = n - k
    duoi = [[x, 0] for x in range(k)]                       # k điểm thẳng hàng ở đáy, x = 0..k-1
    tren = [[x, 2] for x in range(k - 1, k - 1 - m, -1)]    # m điểm thẳng hàng ở đỉnh, x từ k-1 giảm dần
    pts = duoi + tren
    if dent:
        pts[k + m // 2][1] = 1
    return pts

for n in (100, 200, 400, 800):
    pts = thin_rect(n)
    t0 = time.perf_counter()
    kq = brute_sides(pts)
    print(f"n = {n:>3}: {1000 * (time.perf_counter() - t0):7.1f} ms, lồi = {kq}")

n = 100:     1.1 ms, lồi = True
n = 200:     4.2 ms, lồi = True
n = 400:    16.9 ms, lồi = True
n = 800:    67.1 ms, lồi = True


Thời gian tăng gần **4 lần mỗi khi `n` gấp đôi** (đặc trưng của `O(n²)`). Ta cần cách nghĩ tốt hơn.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Chỉ cần nhìn chỗ rẽ.** Với đa giác đơn, "lồi" ⇔ "mọi chỗ rẽ cùng hướng" (định nghĩa số 2). Chỗ rẽ tại đỉnh `P(i+1)` chỉ phụ thuộc **ba đỉnh liên tiếp**: `cross(Pi, P(i+1), P(i+2))`.
2. **Điểm thẳng hàng (tích = 0)** không phải chỗ rẽ nào: **bỏ qua**, nhưng **không** được quên hướng đã thấy trước đó.
3. **Cách làm:** giữ biến `pre` = tích **khác 0** gần nhất. Với mỗi đỉnh: nếu `cur != 0`: nếu `cur * pre < 0` (trái dấu) thì **trả `False`**; ngược lại gán `pre = cur`. Quét `n` đỉnh (kể cả hai đỉnh cuối: dùng chỉ số `% n`).
4. Một lần quét `O(n)`, bộ nhớ `O(1)`.

**Vì sao cần "đa giác đơn":** ngôi sao năm cánh (vẽ một nét) có mọi chỗ rẽ cùng hướng nhưng **không** lồi, vì nó tự cắt. Đề đảm bảo đa giác đơn nên ta khỏi lo.

In [10]:
# Ngôi sao năm cánh (vẽ một nét): mọi chỗ rẽ cùng hướng nhưng KHÔNG đơn, không lồi.
A, B, C, D, E = (0, 10), (10, 3), (6, -9), (-6, -9), (-10, 3)
sao = [A, C, E, B, D]
print(cac_tich(sao))
print("tất cả cùng dấu:", len({t > 0 for t in cac_tich(sao)}) == 1)

[-232, -240, -240, -232, -228]
tất cả cùng dấu: True


## 7. Các cách tiếp theo

### Cách 2 — Dấu tích có hướng của ba đỉnh liên tiếp (cách chuẩn)

**Giải thích từng đoạn:**

- `n = len(points)`; `pre = 0` (chưa thấy hướng nào).
- Với `i` từ `0` đến `n - 1`: lấy `a = points[i]`, `b = points[(i+1) % n]`, `c = points[(i+2) % n]`; `cur = cross(a, b, c)`.
- Nếu `cur != 0`: nếu `cur * pre < 0` → `return False`; rồi `pre = cur`. (Chỉ gán `pre` khi `cur` khác 0: nhớ hướng cuối cùng thật sự.)
- Hết vòng lặp: `return True`.

In [11]:
def cross_signs(points):
    n = len(points)
    pre = 0
    for i in range(n):
        cur = cross(points[i], points[(i + 1) % n], points[(i + 2) % n])
        if cur != 0:
            if cur * pre < 0:
                return False
            pre = cur
    return True

for pts, expected in examples:
    print(pts, cross_signs(pts), expected)
    assert cross_signs(pts) == expected
print(cross_signs([[0, 0], [2, 0], [4, 0], [4, 4], [0, 4]]), "(hình có đỉnh thẳng hàng vẫn lồi)")

[[0, 0], [0, 5], [5, 5], [5, 0]] True True
[[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]] False False
True (hình có đỉnh thẳng hàng vẫn lồi)


In [12]:
class SolutionCross:
    def isConvex(self, points: List[List[int]]) -> bool:
        n = len(points)
        pre = 0
        for i in range(n):
            x1 = points[(i + 1) % n][0] - points[i][0]
            y1 = points[(i + 1) % n][1] - points[i][1]
            x2 = points[(i + 2) % n][0] - points[i][0]
            y2 = points[(i + 2) % n][1] - points[i][1]
            cur = x1 * y2 - x2 * y1
            if cur != 0:
                if cur * pre < 0:
                    return False
                pre = cur
        return True

print(SolutionCross().isConvex([[0, 0], [0, 5], [5, 5], [5, 0]]))

True


**Độ phức tạp (Cách 2):** một lượt `n` bước, mỗi bước `O(1)` → `O(n)` thời gian, `O(1)` bộ nhớ. (Python dùng số nguyên lớn tùy ý nên không lo tràn số; ở C++/Java, tích tối đa cỡ `(2*10^4)² * 2 = 8*10^8` vừa lọt `int` nhưng nên dùng `long`.)

### Cách 3 — So diện tích đa giác với diện tích bao lồi

**Ý tưởng khác:** đa giác đơn luôn nằm **trong** bao lồi của các đỉnh của nó. Nó **lồi** ⇔ nó **trùng** bao lồi ⇔ **diện tích bằng nhau**.

Ta tính bao lồi bằng thuật toán **Andrew (monotone chain)**: sắp các điểm theo `(x, y)`; dựng "chuỗi dưới" rồi "chuỗi trên"; khi thêm điểm mới `p`, nếu ba điểm cuối `h[-2], h[-1], p` **không rẽ trái thật sự** (`cross <= 0`) thì bỏ `h[-1]`. Dùng `<= 0` (bỏ cả điểm thẳng hàng) cho bao lồi "chặt": chỉ giữ các đỉnh thật sự là góc. Diện tích không đổi khi bỏ điểm thẳng hàng nên so sánh vẫn đúng.

In [13]:
def convex_hull(pts):
    pts = sorted(set(map(tuple, pts)))
    if len(pts) <= 2:
        return pts

    def build(seq):
        h = []
        for p in seq:
            while len(h) >= 2 and cross(h[-2], h[-1], p) <= 0:
                h.pop()
            h.append(p)
        return h

    duoi = build(pts)
    tren = build(reversed(pts))
    return duoi[:-1] + tren[:-1]

def hull_area(points):
    return area2(points) == area2(convex_hull(points))

print(convex_hull([[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]]))        # không có (5,5): nó nằm bên trong
for pts, expected in examples:
    print(pts, hull_area(pts), expected, "| gấp đôi diện tích:", area2(pts), "so với hull:", area2(convex_hull(pts)))
    assert hull_area(pts) == expected

[(0, 0), (10, 0), (10, 10), (0, 10)]
[[0, 0], [0, 5], [5, 5], [5, 0]] True True | gấp đôi diện tích: 50 so với hull: 50
[[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]] False False | gấp đôi diện tích: 150 so với hull: 200


In [14]:
class SolutionHull:
    def isConvex(self, points: List[List[int]]) -> bool:
        def cr(a, b, c):
            return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])

        def area2(poly):
            n = len(poly)
            return abs(sum(poly[i][0] * poly[(i + 1) % n][1] - poly[(i + 1) % n][0] * poly[i][1] for i in range(n)))

        pts = sorted(map(tuple, points))

        def build(seq):
            h = []
            for p in seq:
                while len(h) >= 2 and cr(h[-2], h[-1], p) <= 0:
                    h.pop()
                h.append(p)
            return h

        hull = build(pts)[:-1] + build(pts[::-1])[:-1]
        return area2(points) == area2(hull)

print(SolutionHull().isConvex([[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]]))

False


**Độ phức tạp (Cách 3):** sắp xếp `O(n log n)` thời gian, `O(n)` bộ nhớ. Chậm hơn Cách 2 nhưng cho một **góc nhìn độc lập** (dựa vào diện tích, không dựa vào dấu chỗ rẽ), rất hữu ích để đối chiếu.

### Cách 4 — Tổng góc ngoài tuyệt đối bằng 360°

**Ý tưởng khác:** ở mỗi đỉnh có một **góc ngoài** (góc ta phải xoay), dương nếu rẽ trái, âm nếu rẽ phải, tính bằng `atan2(cross, dot)` của hai cạnh liên tiếp. Với **mọi** đa giác đơn, tổng đại số các góc ngoài bằng `±360°` (định lý "tổng góc ngoài"). Nếu mọi góc cùng dấu thì **tổng giá trị tuyệt đối = |tổng| = 360°**; chỉ cần một góc ngược dấu thì tổng giá trị tuyệt đối **lớn hơn** `|tổng|`. Vậy: lồi ⇔ `Σ|θi| = |Σθi|` (sai số số thực nhỏ).

> Đây là cách **dùng số thực**, có sai số; với tọa độ `<= 10^4` góc lệch nhỏ nhất của một chỗ lõm vẫn cỡ `10^-9` rad nên dung sai `10^-10` đủ phân biệt, nhưng đây là cách tham khảo/đối chiếu, **không** nên nộp bài.

In [15]:
def cac_goc_ngoai(points):
    """Danh sách góc ngoài (radian) tại mỗi đỉnh: dương = rẽ trái, âm = rẽ phải."""
    n = len(points)
    ds = []
    for i in range(n):
        a, b, c = points[i], points[(i + 1) % n], points[(i + 2) % n]
        ux, uy = b[0] - a[0], b[1] - a[1]
        vx, vy = c[0] - b[0], c[1] - b[1]
        ds.append(math.atan2(ux * vy - uy * vx, ux * vx + uy * vy))       # trong (-pi, pi]
    return ds

def angle_sum(points, eps=1e-10):
    gs = cac_goc_ngoai(points)
    return sum(abs(g) for g in gs) - abs(sum(gs)) < eps

for pts, expected in examples:
    print(pts, angle_sum(pts), expected)
    assert angle_sum(pts) == expected
nam_giac = [[0, 10], [10, 3], [6, -9], [-6, -9], [-10, 3]]            # ngũ giác lồi
print(angle_sum(nam_giac), "| tổng góc ngoài (độ) =", round(math.degrees(sum(cac_goc_ngoai(nam_giac)))))
print("ví dụ 2: các góc ngoài (độ) =", [round(math.degrees(g)) for g in cac_goc_ngoai(examples[1][0])])

[[0, 0], [0, 5], [5, 5], [5, 0]] True True
[[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]] False False
True | tổng góc ngoài (độ) = -360
ví dụ 2: các góc ngoài (độ) = [-90, -90, -135, 90, -135]


In [16]:
import math as _m

class SolutionAngle:
    def isConvex(self, points: List[List[int]]) -> bool:
        n = len(points)
        s = sa = 0.0
        for i in range(n):
            a, b, c = points[i], points[(i + 1) % n], points[(i + 2) % n]
            ux, uy, vx, vy = b[0] - a[0], b[1] - a[1], c[0] - b[0], c[1] - b[1]
            g = _m.atan2(ux * vy - uy * vx, ux * vx + uy * vy)
            s += g
            sa += abs(g)
        return sa - abs(s) < 1e-10

print(SolutionAngle().isConvex([[0, 0], [0, 5], [5, 5], [5, 0]]))

True


**Độ phức tạp (Cách 4):** `O(n)` thời gian, `O(1)` bộ nhớ nhưng mỗi bước tốn `atan2` (chậm hơn tích có hướng nhiều) và có sai số.

## 8. TRACE từng bước

Trace **Cách 2** (`cross_signs`) với `points = [[0,0],[2,0],[4,0],[4,4],[0,4]]`: hình chữ nhật có thêm đỉnh `P1 = (2,0)` nằm giữa cạnh dưới (thẳng hàng).

| i | `a, b, c` | `cur` | hành động | `pre` sau bước |
|---|---|---|---|---|
| 0 | (0,0),(2,0),(4,0) | `2*0 - 4*0 = 0` | thẳng hàng: bỏ qua, **không đổi `pre`** | 0 |
| 1 | (2,0),(4,0),(4,4) | `2*4 - 2*0 = 8` | `8*0 = 0`, không trái dấu; `pre = 8` | 8 |
| 2 | (4,0),(4,4),(0,4) | `0*4 - (-4)*4 = 16` | cùng dấu; `pre = 16` | 16 |
| 3 | (4,4),(0,4),(0,0) | `(-4)(-4) - (-4)(0) = 16` | cùng dấu; `pre = 16` | 16 |
| 4 | (0,4),(0,0),(2,0) | `(0)(-4) - (2)(-4)` = 8 | cùng dấu; `pre = 8` | 8 |

Không có lần đổi dấu ⇒ `True`. Hàm `trace_convex` bên dưới in đúng những dòng này, rồi chạy thêm Ví dụ 2.

In [17]:
def trace_convex(points):
    n = len(points)
    pre = 0
    for i in range(n):
        a, b, c = points[i], points[(i + 1) % n], points[(i + 2) % n]
        x1, y1 = b[0] - a[0], b[1] - a[1]
        x2, y2 = c[0] - a[0], c[1] - a[1]
        cur = x1 * y2 - x2 * y1
        if cur == 0:
            hanh_dong = "thẳng hàng: bỏ qua"
        elif cur * pre < 0:
            print(f"i={i}: a={a} b={b} c={c} u={(x1, y1)} v={(x2, y2)} cur={cur} pre={pre} -> TRÁI DẤU: return False")
            return False
        else:
            hanh_dong = "cùng dấu (hoặc lần đầu): cập nhật pre"
            pre = cur
        print(f"i={i}: a={a} b={b} c={c} u={(x1, y1)} v={(x2, y2)} cur={cur} -> {hanh_dong}; pre={pre}")
    print("không có đổi dấu: return True")
    return True

trace_convex([[0, 0], [2, 0], [4, 0], [4, 4], [0, 4]])

i=0: a=[0, 0] b=[2, 0] c=[4, 0] u=(2, 0) v=(4, 0) cur=0 -> thẳng hàng: bỏ qua; pre=0
i=1: a=[2, 0] b=[4, 0] c=[4, 4] u=(2, 0) v=(2, 4) cur=8 -> cùng dấu (hoặc lần đầu): cập nhật pre; pre=8
i=2: a=[4, 0] b=[4, 4] c=[0, 4] u=(0, 4) v=(-4, 4) cur=16 -> cùng dấu (hoặc lần đầu): cập nhật pre; pre=16
i=3: a=[4, 4] b=[0, 4] c=[0, 0] u=(-4, 0) v=(-4, -4) cur=16 -> cùng dấu (hoặc lần đầu): cập nhật pre; pre=16
i=4: a=[0, 4] b=[0, 0] c=[2, 0] u=(0, -4) v=(2, -4) cur=8 -> cùng dấu (hoặc lần đầu): cập nhật pre; pre=8
không có đổi dấu: return True


True

In [18]:
# Trace Ví dụ 2: đỉnh lõm (5,5) làm đổi dấu
trace_convex([[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]])

i=0: a=[0, 0] b=[0, 10] c=[10, 10] u=(0, 10) v=(10, 10) cur=-100 -> cùng dấu (hoặc lần đầu): cập nhật pre; pre=-100
i=1: a=[0, 10] b=[10, 10] c=[10, 0] u=(10, 0) v=(10, -10) cur=-100 -> cùng dấu (hoặc lần đầu): cập nhật pre; pre=-100
i=2: a=[10, 10] b=[10, 0] c=[5, 5] u=(0, -10) v=(-5, -5) cur=-50 -> cùng dấu (hoặc lần đầu): cập nhật pre; pre=-50
i=3: a=[10, 0] b=[5, 5] c=[0, 0] u=(-5, 5) v=(-10, 0) cur=50 pre=-50 -> TRÁI DẤU: return False


False

## 9. Code nộp LeetCode

Dùng Cách 2 (dấu tích có hướng của ba đỉnh liên tiếp): `O(n)` thời gian, `O(1)` bộ nhớ.

In [19]:
class Solution:
    def isConvex(self, points: List[List[int]]) -> bool:
        n = len(points)
        pre = 0                                    # dấu (khác 0) của chỗ rẽ gần nhất đã thấy
        for i in range(n):
            x1 = points[(i + 1) % n][0] - points[i][0]
            y1 = points[(i + 1) % n][1] - points[i][1]
            x2 = points[(i + 2) % n][0] - points[i][0]
            y2 = points[(i + 2) % n][1] - points[i][1]
            cur = x1 * y2 - x2 * y1               # tích có hướng của (b - a) và (c - a)
            if cur != 0:                           # thẳng hàng thì bỏ qua
                if cur * pre < 0:                  # đổi hướng rẽ => có đỉnh lõm
                    return False
                pre = cur
        return True


for pts, expected in examples:
    print(pts, Solution().isConvex(pts), expected)
    assert Solution().isConvex(pts) == expected

[[0, 0], [0, 5], [5, 5], [5, 0]] True True
[[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]] False False


## 10. Vì sao thuật toán đúng

**Định lý (đã biết).** Một đa giác **đơn** là lồi ⇔ khi đi vòng quanh biên, mọi chỗ rẽ **không** đổi hướng (mọi tích có hướng khác 0 cùng dấu).

**Phác thảo chứng minh.**

(⇒) Nếu đa giác lồi, nó nằm trọn về một phía của mỗi đường thẳng chứa cạnh. Đi ngược chiều kim đồng hồ thì phần trong luôn ở bên trái, nên các đỉnh kế tiếp luôn nằm bên trái (hoặc trên) cạnh: mọi tích `>= 0`. Theo chiều kia thì mọi tích `<= 0`.

(⇐) Giả sử mọi chỗ rẽ cùng hướng (không đổi dấu). Tổng góc ngoài (có dấu) của đa giác đơn là `±360°`. Vì các góc ngoài cùng dấu và mỗi góc có độ lớn `< 180°`, đường biên **chỉ quay một chiều** và quay đúng một vòng. Đường cong khép kín, quay một chiều đúng một vòng và không tự cắt thì giới hạn một miền lồi. Ta kiểm chứng điều này **bằng thực nghiệm** ở mục 12 với ba cách định nghĩa độc lập (mọi cạnh cùng phía, so diện tích bao lồi, tổng góc ngoài).

**Vai trò của "đa giác đơn":** nếu cho phép tự cắt, tổng góc ngoài có thể là `±720°`... (ngôi sao năm cánh ở mục 6) và định lý sai.

**Vai trò của việc bỏ qua `0`:** tích `0` là ba điểm thẳng hàng (góc đúng 180°) — không phải chỗ rẽ, nên không được phép làm mất `pre`. Nếu `pre` bị gán thành 0, ta "quên" hướng đã thấy và có thể bỏ lọt một lần đổi dấu (xem Lỗi 1).

**Phải xét cả hai đỉnh cuối.** Chỗ rẽ tại `P(n-1)` dùng `P(n-2), P(n-1), P0` và tại `P0` dùng `P(n-1), P0, P1`. Vì vậy vòng lặp chạy đủ `n` bước với chỉ số `% n`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mỗi cạnh, mọi đỉnh cùng phía | `O(n²)` | `O(1)` |
| 2. Dấu tích có hướng ba đỉnh liên tiếp | `O(n)` | `O(1)` |
| 3. So diện tích với bao lồi | `O(n log n)` | `O(n)` |
| 4. Tổng góc ngoài tuyệt đối | `O(n)` | `O(1)` (số thực, có sai số) |

Chặn dưới: phải đọc mọi điểm nên không thể tốt hơn `O(n)`. Với `n = 10^4`, Cách 2 chạy trong vài mili-giây.

## 12. Bộ test

Vì có nhiều cách định nghĩa "lồi" độc lập, ta sẽ **dựng bộ sinh đa giác đơn ngẫu nhiên** (có kiểm tra "đơn" bằng toán học chính xác) và so Cách 2 với Cách 1 (định nghĩa "mọi cạnh cùng phía"), Cách 3, Cách 4.

### 12.1. Ví dụ của đề và edge case

In [20]:
sol = Solution()
f = sol.isConvex

for pts, expected in examples:
    assert f(pts) == expected

assert f([[0, 0], [1, 0], [0, 1]]) is True                                # tam giác luôn lồi
assert f([[0, 0], [0, 1], [1, 0]]) is True                                # tam giác, chiều kim đồng hồ
assert f([[0, 0], [4, 0], [4, 4], [0, 4]]) is True                        # ngược chiều kim đồng hồ
assert f([[0, 0], [0, 4], [4, 4], [4, 0]]) is True                        # cùng chiều kim đồng hồ
assert f([[0, 0], [2, 0], [4, 0], [4, 4], [0, 4]]) is True                # đỉnh thẳng hàng (cạnh dưới)
assert f([[2, 0], [4, 0], [4, 4], [0, 4], [0, 0]]) is True                # cùng hình, bắt đầu ở đỉnh khác
assert f([[0, 0], [4, 0], [4, 4], [2, 2], [0, 4]]) is False               # đỉnh lõm hình chữ V
assert f([[0, 0], [2, 0], [2, 1], [1, 1], [1, 2], [0, 2]]) is False       # hình chữ L
assert f([[-10000, -10000], [10000, -10000], [10000, 10000], [-10000, 10000]]) is True    # biên của ràng buộc
print("Ví dụ và edge case OK")

Ví dụ và edge case OK


### 12.2. Phản ví dụ

Các đa giác này làm vỡ vài lời giải làm ẩu (xem mục 13): chỉ xét `n - 2` đỉnh, coi `0` là một "hướng", đặt lại `pre` về `0`.

In [21]:
def sai_thieu_vong(points):                      # SAI: bỏ hai đỉnh cuối (không quay vòng)
    n = len(points)
    pre = 0
    for i in range(n - 2):
        cur = cross(points[i], points[i + 1], points[i + 2])
        if cur * pre < 0:
            return False
        pre = cur
    return True

def sai_khong_cho_0(points):                     # SAI: đòi mọi tích cùng dấu *nghiêm ngặt* (bắt 0 là một hướng)
    n = len(points)
    cs = [cross(points[i], points[(i + 1) % n], points[(i + 2) % n]) for i in range(n)]
    return all(c > 0 for c in cs) or all(c < 0 for c in cs)

def sai_pre_ve_0(points):                        # SAI: gán pre = cur cả khi cur = 0 (quên hướng)
    n = len(points)
    pre = 0
    for i in range(n):
        cur = cross(points[i], points[(i + 1) % n], points[(i + 2) % n])
        if cur * pre < 0:
            return False
        pre = cur
    return True

# Ví dụ 2 vỡ lời giải thiếu vòng: lõm nằm ở hai đỉnh cuối
p2 = [[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]]
print("thiếu vòng:", sai_thieu_vong(p2), "(đúng: False)")
# Hình có đỉnh thẳng hàng vỡ lời giải đòi nghiêm ngặt
p3 = [[0, 0], [2, 0], [4, 0], [4, 4], [0, 4]]
print("đòi nghiêm ngặt:", sai_khong_cho_0(p3), "(đúng: True)")
# Chữ L có đỉnh thẳng hàng ngay trước đỉnh lõm, và đỉnh lõm đặt ở vị trí 0
p4 = [[2, 2], [2, 4], [0, 4], [0, 0], [4, 0], [4, 2], [3, 2]]
print("pre về 0:", sai_pre_ve_0(p4), "(đúng: False)")
assert sai_thieu_vong(p2) is True and f(p2) is False
assert sai_khong_cho_0(p3) is False and f(p3) is True
assert sai_pre_ve_0(p4) is True and f(p4) is False

thiếu vòng: True (đúng: False)
đòi nghiêm ngặt: False (đúng: True)
pre về 0: True (đúng: False)


### 12.3. Bộ sinh đa giác đơn ngẫu nhiên (có seed) và kiểm tra "đơn" chính xác

Để test có ý nghĩa, đầu vào phải là **đa giác đơn**. Ta cần một hàm `la_don(points)` kiểm tra đúng nghĩa (số nguyên, chính xác):

- không có điểm trùng;
- hai cạnh **không kề nhau** không được giao nhau (kể cả chạm đầu);
- hai cạnh **kề nhau** chỉ được chung đúng đỉnh chung (nếu thẳng hàng thì không được gập ngược lại).

In [22]:
def dau(x):
    return (x > 0) - (x < 0)

def trong_doan(a, b, p):
    """p (đã thẳng hàng với a, b) có nằm trong đoạn [a, b] không."""
    return min(a[0], b[0]) <= p[0] <= max(a[0], b[0]) and min(a[1], b[1]) <= p[1] <= max(a[1], b[1])

def doan_giao(a, b, c, d):
    """Hai đoạn ab và cd có điểm chung không (kể cả chạm đầu)."""
    d1, d2 = dau(cross(a, b, c)), dau(cross(a, b, d))
    d3, d4 = dau(cross(c, d, a)), dau(cross(c, d, b))
    if d1 * d2 < 0 and d3 * d4 < 0:
        return True
    if d1 == 0 and trong_doan(a, b, c): return True
    if d2 == 0 and trong_doan(a, b, d): return True
    if d3 == 0 and trong_doan(c, d, a): return True
    if d4 == 0 and trong_doan(c, d, b): return True
    return False

def la_don(points):
    n = len(points)
    if len({tuple(p) for p in points}) != n:
        return False
    for i in range(n):
        for j in range(i + 1, n):
            if j == i + 1:                                     # hai cạnh kề nhau, chung đỉnh points[j]
                S, A, B = points[j], points[i], points[(j + 1) % n]
            elif i == 0 and j == n - 1:                        # cạnh đầu và cạnh cuối, chung đỉnh points[0]
                S, A, B = points[0], points[1], points[n - 1]
            else:                                              # hai cạnh không kề nhau: không được có điểm chung
                if doan_giao(points[i], points[(i + 1) % n], points[j], points[(j + 1) % n]):
                    return False
                continue
            if cross(S, A, B) == 0 and (A[0] - S[0]) * (B[0] - S[0]) + (A[1] - S[1]) * (B[1] - S[1]) > 0:
                return False                                   # thẳng hàng và cùng phía: hai cạnh chồng lên nhau
    return True

print(la_don([[0, 0], [0, 5], [5, 5], [5, 0]]), la_don([[0, 0], [5, 5], [0, 5], [5, 0]]), la_don(sao))   # True False False
assert la_don([[0, 0], [0, 5], [5, 5], [5, 0]]) is True
assert la_don([[0, 0], [5, 5], [0, 5], [5, 0]]) is False          # hình nơ: hai cạnh cắt nhau
assert la_don([list(p) for p in sao]) is False                    # ngôi sao năm cánh tự cắt
assert la_don([[0, 0], [2, 0], [4, 0], [4, 4], [0, 4]]) is True   # đỉnh thẳng hàng vẫn đơn

True False False


In [23]:
random.seed(469)

def sinh_da_giac_don(n_max=8, R=5):
    """Chọn ngẫu nhiên n điểm trên lưới [-R, R]^2, hoán vị ngẫu nhiên, giữ lại nếu là đa giác đơn."""
    while True:
        n = random.randint(3, n_max)
        pts = random.sample([[x, y] for x in range(-R, R + 1) for y in range(-R, R + 1)], n)
        if la_don(pts):
            return pts

dem = {True: 0, False: 0}
for _ in range(3000):
    pts = sinh_da_giac_don()
    oracle = brute_sides(pts)                  # định nghĩa "mọi cạnh cùng phía"
    dem[oracle] += 1
    assert f(pts) == oracle, pts
    assert SolutionCross().isConvex(pts) == oracle, pts
    assert hull_area(pts) == oracle, pts
    assert SolutionHull().isConvex(pts) == oracle, pts
    assert angle_sum(pts) == oracle, pts
    assert SolutionAngle().isConvex(pts) == oracle, pts
print("3000 đa giác đơn ngẫu nhiên: Cách 1 = 2 = 3 = 4. Phân bố (lồi / không lồi):", dem[True], "/", dem[False])
assert min(dem.values()) > 100

3000 đa giác đơn ngẫu nhiên: Cách 1 = 2 = 3 = 4. Phân bố (lồi / không lồi): 2015 / 985


### 12.4. Sinh đa giác lồi chắc chắn (kể cả có đỉnh thẳng hàng, mọi chiều quay, mọi điểm bắt đầu)

Lấy bao lồi chặt của các điểm ngẫu nhiên (chắc chắn lồi), thêm **các điểm thẳng hàng** trên cạnh (điểm nguyên nằm giữa hai đỉnh), đảo chiều quay ngẫu nhiên và quay vòng điểm bắt đầu. Kết quả đúng phải là `True`. Sau đó **"phá" lồi** bằng cách kéo một đỉnh sâu vào trong (nếu vẫn là đa giác đơn thì đáp án thường là `False`; ta kiểm tra bằng Cách 1).

In [24]:
from math import gcd

def sinh_loi(R=12):
    while True:
        pts = random.sample([(x, y) for x in range(-R, R + 1) for y in range(-R, R + 1)], random.randint(3, 12))
        h = convex_hull(pts)
        if len(h) >= 3:
            break
    poly = []
    for i, a in enumerate(h):
        b = h[(i + 1) % len(h)]
        poly.append(list(a))
        g = gcd(abs(b[0] - a[0]), abs(b[1] - a[1]))
        for k in range(1, g):                                   # điểm nguyên thẳng hàng nằm giữa a và b
            if random.random() < 0.5:
                poly.append([a[0] + (b[0] - a[0]) // g * k, a[1] + (b[1] - a[1]) // g * k])
    if random.random() < 0.5:
        poly.reverse()
    s = random.randrange(len(poly))
    return poly[s:] + poly[:s]

random.seed(4690)
so_co_thang_hang = 0
for _ in range(1500):
    poly = sinh_loi()
    assert la_don(poly), poly
    assert f(poly) is True, poly
    assert brute_sides(poly) and hull_area(poly) and angle_sum(poly)
    so_co_thang_hang += any(c == 0 for c in cac_tich(poly))
print("1500 đa giác lồi: đều True ở mọi cách; số đa giác có đỉnh thẳng hàng:", so_co_thang_hang)
assert so_co_thang_hang > 100

1500 đa giác lồi: đều True ở mọi cách; số đa giác có đỉnh thẳng hàng: 1163


### 12.5. Test lớn (ràng buộc `n = 10^4`)

Hình chữ nhật mỏng có `10^4` đỉnh, hầu hết thẳng hàng (lồi), và biến thể có một vết lõm (không lồi).

In [25]:
lon = thin_rect(10_000)
lon_lom = thin_rect(10_000, dent=True)
t0 = time.perf_counter()
r1, r2 = f(lon), f(lon_lom)
print(r1, r2, f"{1000 * (time.perf_counter() - t0):.1f} ms")
assert r1 is True and r2 is False

# Kiểm chứng tính chất của thin_rect ở kích thước nhỏ bằng oracle
for n in (4, 5, 6, 9, 10, 31, 40):
    a, b = thin_rect(n), thin_rect(n, dent=True)
    assert la_don(a) and la_don(b)
    assert brute_sides(a) is True and f(a) is True
    if n >= 6:
        assert brute_sides(b) is False and f(b) is False
print("thin_rect OK; tọa độ trong khoảng", min(p[0] for p in lon), "..", max(p[0] for p in lon))

True False 4.4 ms
thin_rect OK; tọa độ trong khoảng 0 .. 4999


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gán `pre = cur` cả khi `cur == 0` (quên hướng đã thấy)

Khi gặp điểm thẳng hàng, `pre` bị xóa về `0`, và lần đổi dấu ngay sau đó không bị phát hiện. Ví dụ: chữ L với một đỉnh thẳng hàng ngay trước đỉnh lõm, được viết để đỉnh lõm là đỉnh cuối cùng của vòng quét.

In [26]:
p4 = [[2, 2], [2, 4], [0, 4], [0, 0], [4, 0], [4, 2], [3, 2]]
print("các tích :", cac_tich(p4))
print("sai :", sai_pre_ve_0(p4), "(mong đợi False)")
print("đúng:", Solution().isConvex(p4))
assert sai_pre_ve_0(p4) is True and Solution().isConvex(p4) is False and brute_sides(p4) is False

các tích : [4, 8, 16, 8, 2, 0, -2]
sai : True (mong đợi False)
đúng: False


**Vì sao sai:** dãy tích là `[+, +, +, +, +, 0, -]`: tại bước `0` ta đặt `pre = 0`, rồi `cur = -` không bị so với `+` trước đó nữa; và lần đổi dấu quay vòng (`-` sang `+` đầu dãy) không được so sánh vì vòng quét kết thúc. **Cách đúng:** chỉ cập nhật `pre` khi `cur != 0`.

### Lỗi 2 — Không quay vòng (bỏ hai đỉnh cuối)

Dùng `range(n - 2)` thì bỏ sót chỗ rẽ tại `P(n-1)` và `P0`. Ví dụ 2 của đề có đỉnh lõm ở vị trí cuối.

In [27]:
print("sai :", sai_thieu_vong(p2), "(mong đợi False)")
print("đúng:", Solution().isConvex(p2))
assert sai_thieu_vong(p2) is True and Solution().isConvex(p2) is False

sai : True (mong đợi False)
đúng: False


**Vì sao sai:** đa giác là một **vòng**: đỉnh cuối nối với đỉnh đầu. **Cách đúng:** chạy `n` bước và dùng `% n`.

### Lỗi 3 — Đòi mọi tích cùng dấu nghiêm ngặt (coi `0` là một hướng)

`all(c > 0)` hoặc `all(c < 0)` loại sai các đa giác lồi có đỉnh thẳng hàng.

In [28]:
print("sai :", sai_khong_cho_0(p3), "(mong đợi True)", cac_tich(p3))
print("đúng:", Solution().isConvex(p3))
assert sai_khong_cho_0(p3) is False and Solution().isConvex(p3) is True

sai : False (mong đợi True) [0, 8, 16, 16, 8]
đúng: True


**Vì sao sai:** ba điểm thẳng hàng cho tích `0`, nhưng góc tại đó đúng 180° — vẫn hợp lệ cho đa giác lồi (theo định nghĩa trong đề, hình chữ nhật có thêm đỉnh ở giữa cạnh vẫn lồi). **Cách đúng:** bỏ qua tích `0`.

### Lỗi 4 — So sánh dấu bằng phép chia/nhân khi số lớn (ở ngôn ngữ khác) hoặc dùng số thực

Trong Python số nguyên không tràn, nhưng ở C++/Java `cur * pre` có thể **tràn số** (tích hai số cỡ `10^9`). Cách an toàn: so sánh **dấu** (`cur > 0` và `pre < 0`...) thay vì nhân. Bản Python tương đương:

In [29]:
def isConvex_dau(points):
    n = len(points)
    co_duong = co_am = False
    for i in range(n):
        cur = cross(points[i], points[(i + 1) % n], points[(i + 2) % n])
        if cur > 0:
            co_duong = True
        elif cur < 0:
            co_am = True
        if co_duong and co_am:
            return False
    return True

random.seed(1)
for _ in range(500):
    pts = sinh_da_giac_don()
    assert isConvex_dau(pts) == f(pts)
print("So sánh theo dấu (không nhân) cho kết quả y hệt, và an toàn hơn khi sang C++/Java")

So sánh theo dấu (không nhân) cho kết quả y hệt, và an toàn hơn khi sang C++/Java


**Vì sao nên:** đỡ lo tràn số và tương đương về logic: chỉ cần biết **có cả tích dương và tích âm hay không**.

### Lưu ý (không phải lỗi code) — đa giác tự cắt

Đề đảm bảo đa giác đơn; nếu bạn tự thử với đa giác tự cắt như ngôi sao năm cánh (mục 6), thuật toán trả `True` nhưng đó không phải lồi: nằm ngoài giả thiết của bài.

In [30]:
print("ngôi sao 5 cánh:", Solution().isConvex([list(p) for p in sao]), "| có phải đơn?", la_don([list(p) for p in sao]))

ngôi sao 5 cánh: True | có phải đơn? False


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "có lồi không / đổi hướng chưa / điểm nằm trái hay phải đường thẳng" → nghĩ ngay tới **tích có hướng**.
- **Công thức thuộc lòng:** `cross(a, b, c) = (bx-ax)*(cy-ay) - (by-ay)*(cx-ax)`. Dương = rẽ trái, âm = rẽ phải, `0` = thẳng hàng.
- **Tích có hướng tránh được chia và số thực:** mọi thứ nguyên, không sai số làm tròn.
- **Quay vòng:** với đa giác, luôn dùng `% n` để nối cuối với đầu.
- **Nhớ "bỏ qua 0 nhưng giữ `pre`":** mẫu "nhớ giá trị khác 0 gần nhất" rất hay gặp.
- **Đa giác lồi ⇔ trùng bao lồi:** nếu phải làm bài về bao lồi, nhớ thuật toán Andrew (sắp xếp + pop khi không rẽ trái).
- **Test hình học nên có:** thẳng hàng, chiều kim đồng hồ và ngược lại, bắt đầu ở đỉnh bất kỳ, đa giác tối thiểu (tam giác).

## 15. Biến thể

**Biến thể A — đếm số đỉnh lõm (reflex).** Quyết định chiều của đa giác bằng dấu diện tích có dấu (shoelace không lấy trị tuyệt đối); đỉnh lõm là chỗ rẽ ngược chiều đó.

**Biến thể B — kiểm tra điểm có nằm trong đa giác lồi không.** Điểm `p` nằm trong (hoặc trên biên) đa giác lồi ngược chiều kim đồng hồ ⇔ `cross(Pi, P(i+1), p) >= 0` với mọi cạnh.

**Biến thể C — diện tích bao lồi / chu vi bao lồi** của một tập điểm: dùng `convex_hull` ở Cách 3.

In [31]:
def dem_dinh_lom(points):
    n = len(points)
    s = sum(points[i][0] * points[(i + 1) % n][1] - points[(i + 1) % n][0] * points[i][1] for i in range(n))
    chieu = 1 if s > 0 else -1                      # +1: ngược chiều kim đồng hồ
    return sum(1 for i in range(n)
               if cross(points[i], points[(i + 1) % n], points[(i + 2) % n]) * chieu < 0)

def diem_trong_loi(poly, p):
    n = len(poly)
    s = sum(poly[i][0] * poly[(i + 1) % n][1] - poly[(i + 1) % n][0] * poly[i][1] for i in range(n))
    chieu = 1 if s > 0 else -1
    return all(cross(poly[i], poly[(i + 1) % n], p) * chieu >= 0 for i in range(n))

print(dem_dinh_lom([[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]]))      # 1 (đỉnh (5,5))
print(dem_dinh_lom([[0, 0], [2, 0], [2, 1], [1, 1], [1, 2], [0, 2]]))   # 1 (đỉnh (1,1))
print(diem_trong_loi([[0, 0], [4, 0], [4, 4], [0, 4]], (2, 2)), diem_trong_loi([[0, 0], [4, 0], [4, 4], [0, 4]], (5, 2)))
assert dem_dinh_lom([[0, 0], [0, 10], [10, 10], [10, 0], [5, 5]]) == 1
assert dem_dinh_lom([[0, 0], [0, 5], [5, 5], [5, 0]]) == 0
assert diem_trong_loi([[0, 0], [4, 0], [4, 4], [0, 4]], (2, 2)) is True
assert diem_trong_loi([[0, 0], [4, 0], [4, 4], [0, 4]], (5, 2)) is False

1
1
True False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cạnh cùng phía | Định nghĩa trực tiếp: các đỉnh khác cùng phía đường thẳng của cạnh | `O(n²)` | `O(1)` | Oracle để đối chiếu, `n` nhỏ |
| 2. Dấu ba đỉnh liên tiếp | Mọi chỗ rẽ cùng hướng, bỏ qua thẳng hàng | `O(n)` | `O(1)` | **Cách nộp bài** |
| 3. Diện tích = diện tích bao lồi | Đa giác lồi ⇔ trùng bao lồi | `O(n log n)` | `O(n)` | Khi đã cần bao lồi; góc nhìn độc lập |
| 4. Tổng góc ngoài tuyệt đối | `Σ|θ| = |Σθ| = 360°` | `O(n)` | `O(1)` | Minh họa hình học; số thực nên không nộp |

In [32]:
lon = thin_rect(2000)
cac = [("Cách 1 brute_sides", brute_sides), ("Cách 2 cross_signs", cross_signs), ("Cách 3 hull_area", hull_area), ("Cách 4 angle_sum", angle_sum)]
for ten, fn in cac:
    t0 = time.perf_counter()
    r = fn(lon)
    print(f"{ten:<20} n = {len(lon)}: {1000 * (time.perf_counter() - t0):8.1f} ms  -> {r}")

Cách 1 brute_sides   n = 2000:    384.2 ms  -> True
Cách 2 cross_signs   n = 2000:      0.3 ms  -> True
Cách 3 hull_area     n = 2000:      1.8 ms  -> True
Cách 4 angle_sum     n = 2000:      0.5 ms  -> True


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán dấu (`+`, `-` hay `0`) của: (a) `cross((0,0),(1,0),(1,1))`; (b) `cross((0,0),(1,1),(2,2))`; (c) `cross((0,0),(1,0),(1,-1))`; (d) `Solution().isConvex([[0,0],[1,0],[0,1]])`.

**Bài 2 (tính tay).** Với `points = [[0,0],[4,0],[4,4],[2,1],[0,4]]`, tính `cur` tại từng `i = 0..4` (như bảng ở mục 8), và cho biết ở bước nào thuật toán trả `False`.

**Bài 3 (điền TODO).** Hoàn thành `is_convex_todo(points)` theo Cách 2.

In [33]:
def is_convex_todo(points):
    # TODO 1: n = len(points); pre = 0
    # TODO 2: for i in range(n): lấy a, b, c = points[i], points[(i+1)%n], points[(i+2)%n]
    # TODO 3: cur = (b[0]-a[0])*(c[1]-a[1]) - (b[1]-a[1])*(c[0]-a[0])
    # TODO 4: nếu cur != 0: nếu cur * pre < 0 -> return False; rồi pre = cur
    # TODO 5: return True
    return None

# KHÔNG gọi is_convex_todo ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `kiem_tra_cach_1(points)` theo Cách 1 (mỗi cạnh, mọi đỉnh cùng phía) **mà không nhìn lại** mục 5. Kiểm tra hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Viết `huong_da_giac(points)` trả `"CCW"` (ngược chiều kim đồng hồ), `"CW"` (cùng chiều) cho đa giác đơn; dùng dấu của diện tích có dấu (shoelace không `abs`). Thử trên `[[0,0],[4,0],[4,4],[0,4]]` (CCW) và đảo ngược của nó (CW). Sau đó viết `dien_tich_co_dau` để dùng lại.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) đi sang phải rồi lên: rẽ trái: dương. (b) ba điểm trên đường chéo: `0`. (c) rẽ phải: âm. (d) mọi tam giác đều lồi. Bài 3: viết đúng 5 TODO theo thứ tự.
- **Mức 2 (vừa):** Bài 2: vết lõm là điểm `[2,1]`; tính `cross` tại `i = 1` (`(4,0),(4,4),(2,1)`) và `i = 2`... Tìm lần đầu hai tích trái dấu. Bài 4: duyệt mọi cạnh `i`, rồi mọi điểm `c`, theo dõi `co_duong` và `co_am`.
- **Mức 3 (sâu):** Bài 5: diện tích có dấu `S = Σ (xi*y(i+1) - x(i+1)*yi)`; `S > 0` ⇒ CCW, `S < 0` ⇒ CW. Với tam giác `(0,0),(4,0),(0,3)`, `S = 12 > 0`.

## 19. Đáp án tham khảo cho bài tập

In [34]:
# Bài 1
assert cross((0, 0), (1, 0), (1, 1)) > 0
assert cross((0, 0), (1, 1), (2, 2)) == 0
assert cross((0, 0), (1, 0), (1, -1)) < 0
assert Solution().isConvex([[0, 0], [1, 0], [0, 1]]) is True
print("Bài 1 OK")

# Bài 2
b2 = [[0, 0], [4, 0], [4, 4], [2, 1], [0, 4]]
print("Bài 2: các tích =", cac_tich(b2))
# i=0: (0,0),(4,0),(4,4): 4*4 - 0*4 = 16 ; i=1: (4,0),(4,4),(2,1): 0*1 - (-2)*4 = 8 ;
# i=2: (4,4),(2,1),(0,4): (-2)*0 - (-4)*(-3) = -12  -> trái dấu với pre = 8 => return False tại i = 2
assert cac_tich(b2)[:3] == [16, 8, -12]
assert Solution().isConvex(b2) is False
print("Bài 2 OK: trả False ở i = 2")

# Bài 3
def is_convex_todo_ans(points):
    n = len(points)
    pre = 0
    for i in range(n):
        a, b, c = points[i], points[(i + 1) % n], points[(i + 2) % n]
        cur = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
        if cur != 0:
            if cur * pre < 0:
                return False
            pre = cur
    return True

random.seed(8)
for _ in range(300):
    pts = sinh_da_giac_don()
    assert is_convex_todo_ans(pts) == Solution().isConvex(pts)
print("Bài 3 OK")

# Bài 4
def kiem_tra_cach_1(points):
    n = len(points)
    for i in range(n):
        a, b = points[i], points[(i + 1) % n]
        co_duong = co_am = False
        for c in points:
            t = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
            co_duong |= t > 0
            co_am |= t < 0
        if co_duong and co_am:
            return False
    return True

for pts, expected in examples:
    assert kiem_tra_cach_1(pts) == expected
for _ in range(300):
    pts = sinh_da_giac_don()
    assert kiem_tra_cach_1(pts) == Solution().isConvex(pts)
print("Bài 4 OK")

# Bài 5
def dien_tich_co_dau(points):
    n = len(points)
    return sum(points[i][0] * points[(i + 1) % n][1] - points[(i + 1) % n][0] * points[i][1] for i in range(n))

def huong_da_giac(points):
    return "CCW" if dien_tich_co_dau(points) > 0 else "CW"

sq = [[0, 0], [4, 0], [4, 4], [0, 4]]
assert huong_da_giac(sq) == "CCW" and huong_da_giac(sq[::-1]) == "CW"
assert dien_tich_co_dau([[0, 0], [4, 0], [0, 3]]) == 12
print("Bài 5 OK")

Bài 1 OK
Bài 2: các tích = [16, 8, -12, 8, 16]
Bài 2 OK: trả False ở i = 2
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (hình học: các chỗ rẽ cùng hướng):**

```python
n = len(points)
pre = 0
for i in range(n):
    a, b, c = points[i], points[(i + 1) % n], points[(i + 2) % n]
    cur = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
    if cur != 0:
        if cur * pre < 0:
            return False
        pre = cur
return True
```

**Checklist:**

- [ ] Tích có hướng đúng công thức `(b-a) × (c-a)`; dương = rẽ trái, âm = rẽ phải, `0` = thẳng hàng.
- [ ] Quét đủ `n` đỉnh với chỉ số `% n` (không bỏ hai đỉnh cuối).
- [ ] Bỏ qua `cur == 0` **nhưng không đặt lại `pre`**.
- [ ] Chỉ cập nhật `pre = cur` khi `cur != 0`.
- [ ] Hiểu vì sao cần giả thiết "đa giác đơn" (ngôi sao năm cánh phản ví dụ).
- [ ] Thử tay: hình vuông (`True`), ví dụ 2 (`False`), hình có đỉnh thẳng hàng (`True`), chữ L (`False`), tam giác (`True`).
- [ ] Đúng signature `class Solution` / `isConvex(self, points: List[List[int]]) -> bool`.